In [1]:
import os
import copy

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

import ibicus
from ibicus.debias import LinearScaling

from tqdm import tqdm

In [2]:
regions = ["Alberta", "Amazon", "Congo", "LA", "NWIndia", "Pantanal"]

In [3]:
scenarios = ['historical', 'ssp126', 'ssp370', 'ssp585']
models = ['GFDL-ESM4', 'IPSL-CM6A-LR', 'MPI-ESM1-2-HR', 'MRI-ESM2-0', 'UKESM1-0-LL']

# Bias adjustment

Implement the bias adjustment method using the ibicus package. 

### Tree cover

#### isimip3b, scenario, model, region:

In [4]:
for region in regions:
    
    tree_cover_jules_hist = xr.open_dataset(f"data/{region}/isimp3a/obsclim/GSWP3-W5E5/period_2002_2019/tree_cover_jules-es.nc")
    tree_cover_obs = xr.open_dataset(f"data/{region}/isimp3a/obsclim/GSWP3-W5E5/period_2002_2019/tree_raw_VCF-obs.nc")
    
    # Extract values
    tree_cover_jules_hist_np = tree_cover_jules_hist.tree_cover.values
    tree_cover_obs_np = tree_cover_obs.variable.values
    
    # Convert NaNs to np.nan (sometimes these are casted to high values)
    tree_cover_jules_hist_np[tree_cover_jules_hist_np > 1e6] = np.nan
    tree_cover_obs_np[tree_cover_obs_np > 1e6] = np.nan
    
    # Convert obs values
    #tree_cover_obs_np = tree_cover_obs_np*100

    # Common nan grid
    common_nan_grid_tree_cover = np.logical_or(np.all(np.isnan(tree_cover_jules_hist_np), axis = 0), np.all(np.isnan(tree_cover_obs_np), axis = 0))
    tree_cover_jules_hist_np[:, common_nan_grid_tree_cover] = np.nan
    tree_cover_obs_np[:, common_nan_grid_tree_cover] = np.nan

    print(f"--------Bias adjusting region {region}----------\n")

    for scenario in tqdm(scenarios):
        for model in models:
            try:
                if scenario == "historical":
                    period = "period_1994_2014"
                else:
                    period = "period_2015_2099"

                # Read in values
                filename = os.path.join("data", region, "isimp3b", scenario, model, period, "tree_cover_jules-es.nc")
                tree_cover_cm_fut = xr.open_dataset(filename)
                
                tree_cover_cm_fut_np = tree_cover_cm_fut.tree_cover.values
                tree_cover_cm_fut_np[tree_cover_cm_fut_np > 1e6] = np.nan

                # Get common nan mask (land-sea mask)
                nan_grid_cm_future = np.all(np.isnan(tree_cover_cm_fut_np), axis = 0)

                # Get values in cm future where values exist, but the common mask indicates nan (to infill later)
                cm_future_not_nan_but_obs_or_cm_hist = np.logical_and(np.logical_not(nan_grid_cm_future), common_nan_grid_tree_cover)

                # Map to common nan mask
                tree_cover_cm_fut_np_common_nan_grid = copy.deepcopy(tree_cover_cm_fut_np)
                tree_cover_cm_fut_np_common_nan_grid[:, common_nan_grid_tree_cover] = np.nan

                # Apply bias correction
                debiaser = LinearScaling(delta_type = "additive")
                debiased_tree_cover_cm_fut_common_nan_grid = debiaser.apply(tree_cover_obs_np, tree_cover_jules_hist_np, tree_cover_cm_fut_np_common_nan_grid,  progressbar = False, failsafe = True)
            
                # Reinfill values in cm_future outside of common nan grid 
                debiased_tree_cover_cm_fut = debiased_tree_cover_cm_fut_common_nan_grid
                debiased_tree_cover_cm_fut[:, cm_future_not_nan_but_obs_or_cm_hist] = tree_cover_cm_fut_np[:, cm_future_not_nan_but_obs_or_cm_hist] 

                # Map values below 0 and above 100 to min, max of obs:
                debiased_tree_cover_cm_fut[debiased_tree_cover_cm_fut < 0] = tree_cover_obs_np[~np.isnan(tree_cover_obs_np)].min()
                debiased_tree_cover_cm_fut[debiased_tree_cover_cm_fut > 100] = tree_cover_obs_np[~np.isnan(tree_cover_obs_np)].max()

                # Insert back into array
                tree_cover_cm_fut.tree_cover.values = debiased_tree_cover_cm_fut

                # Write to file
                tree_cover_cm_fut.to_netcdf(os.path.join("data", region, "isimp3b", scenario, model, period, "debiased_tree_cover_jules-es.nc"))


            except Exception as e:
                print(region, scenario, model)
                print(e)


--------Bias adjusting region Alberta----------



  0%|                                                     | 0/4 [00:00<?, ?it/s]/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: obs contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_hist contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_future contains inf or nan values. Not all debiasers support missing valu

--------Bias adjusting region Amazon----------



  0%|                                                     | 0/4 [00:00<?, ?it/s]/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: obs contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_hist contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_future contains inf or nan values. Not all debiasers support missing valu

--------Bias adjusting region Congo----------



...

  0%|                                                     | 0/4 [00:00<?, ?it/s]/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: obs contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_hist contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_future contains inf or nan values. Not all debiasers support missing valu

--------Bias adjusting region NWIndia----------



  0%|                                                     | 0/4 [00:00<?, ?it/s]/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: obs contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_hist contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_future contains inf or nan values. Not all debiasers support missing valu

--------Bias adjusting region Pantanal----------



  0%|                                                     | 0/4 [00:00<?, ?it/s]/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: obs contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_hist contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_future contains inf or nan values. Not all debiasers support missing valu

#### isimip3a, obsclim or counterclim, GSWP3-W5E5:

In [27]:
for region in regions:
    
    tree_cover_jules_hist = xr.open_dataset(f"data/{region}/isimp3a/obsclim/GSWP3-W5E5/period_2002_2019/tree_cover_jules-es.nc")
    tree_cover_obs = xr.open_dataset(f"data/{region}/isimp3a/obsclim/GSWP3-W5E5/period_2002_2019/tree_raw_VCF-obs.nc")
    
    # Extract values
    tree_cover_jules_hist_np = tree_cover_jules_hist.tree_cover.values
    tree_cover_obs_np = tree_cover_obs.variable.values
    
    # Convert NaNs to np.nan (sometimes these are casted to high values)
    tree_cover_jules_hist_np[tree_cover_jules_hist_np > 1e6] = np.nan
    tree_cover_obs_np[tree_cover_obs_np > 1e6] = np.nan
    
    # Convert obs values
    #tree_cover_obs_np = tree_cover_obs_np*100

    # Common nan grid
    common_nan_grid_tree_cover = np.logical_or(np.all(np.isnan(tree_cover_jules_hist_np), axis = 0), np.all(np.isnan(tree_cover_obs_np), axis = 0))
    tree_cover_jules_hist_np[:, common_nan_grid_tree_cover] = np.nan
    tree_cover_obs_np[:, common_nan_grid_tree_cover] = np.nan

    print(f"--------Bias adjusting region {region}----------\n")

    for clim_type in ["obsclim", "counterclim"]:
        for period in ["period_2000_2019", "period_1901_1920"]:
            
            try:
                
                # Read in values
                filename = os.path.join("data", region, "isimp3a", clim_type, "GSWP3-W5E5", period, "tree_cover_jules-es.nc")
                tree_cover_cm_fut = xr.open_dataset(filename)
                
                tree_cover_cm_fut_np = tree_cover_cm_fut.tree_cover.values
                tree_cover_cm_fut_np[tree_cover_cm_fut_np > 1e6] = np.nan

                # Get common nan mask (land-sea mask)
                nan_grid_cm_future = np.all(np.isnan(tree_cover_cm_fut_np), axis = 0)

                # Get values in cm future where values exist, but the common mask indicates nan (to infill later)
                cm_future_not_nan_but_obs_or_cm_hist = np.logical_and(np.logical_not(nan_grid_cm_future), common_nan_grid_tree_cover)

                # Map to common nan mask
                tree_cover_cm_fut_np_common_nan_grid = copy.deepcopy(tree_cover_cm_fut_np)
                tree_cover_cm_fut_np_common_nan_grid[:, common_nan_grid_tree_cover] = np.nan

                # Apply bias correction
                debiaser = LinearScaling(delta_type = "additive")
                debiased_tree_cover_cm_fut_common_nan_grid = debiaser.apply(tree_cover_obs_np, tree_cover_jules_hist_np, tree_cover_cm_fut_np_common_nan_grid,  progressbar = False, failsafe = True)
            
                # Reinfill values in cm_future outside of common nan grid 
                debiased_tree_cover_cm_fut = debiased_tree_cover_cm_fut_common_nan_grid
                debiased_tree_cover_cm_fut[:, cm_future_not_nan_but_obs_or_cm_hist] = tree_cover_cm_fut_np[:, cm_future_not_nan_but_obs_or_cm_hist] 

                # Map values below 0 and above 100 to min, max of obs:
                debiased_tree_cover_cm_fut[debiased_tree_cover_cm_fut < 0] = tree_cover_obs_np[~np.isnan(tree_cover_obs_np)].min()
                debiased_tree_cover_cm_fut[debiased_tree_cover_cm_fut > 100] = tree_cover_obs_np[~np.isnan(tree_cover_obs_np)].max()

                # Insert back into array
                tree_cover_cm_fut.tree_cover.values = debiased_tree_cover_cm_fut

                # Write to file
                tree_cover_cm_fut.to_netcdf(os.path.join("data", region, "isimp3a", clim_type, "GSWP3-W5E5", period, "debiased_tree_cover_jules-es.nc"))

            except Exception as e:
                print(region, clim_type, period)
                print(e)

--------Bias adjusting region Alberta----------

--------Bias adjusting region Amazon----------



/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: obs contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_hist contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_future contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. 

--------Bias adjusting region Congo----------



/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: obs contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_hist contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_future contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. 

--------Bias adjusting region LA----------

--------Bias adjusting region NWIndia----------



/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: obs contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_hist contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_future contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. 

--------Bias adjusting region Pantanal----------



/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: obs contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_hist contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_future contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. 

## Nonetree cover

isimip3b, scenario, model, region:

In [36]:
for region in regions:
    
    nonetree_cover_jules_hist = xr.open_dataset(f"data/{region}/isimp3a/obsclim/GSWP3-W5E5/period_2002_2019/nonetree_cover_jules-es.nc")
    nonetree_cover_obs = xr.open_dataset(f"data/{region}/isimp3a/obsclim/GSWP3-W5E5/period_2002_2019/nontree_raw_VCF-obs.nc")
    
    # Extract values
    nonetree_cover_jules_hist_np = nonetree_cover_jules_hist.nonetree_cover.values
    nonetree_cover_obs_np = nonetree_cover_obs.variable.values
    
    # Convert NaNs to np.nan (sometimes these are casted to high values)
    nonetree_cover_jules_hist_np[nonetree_cover_jules_hist_np > 1e6] = np.nan
    nonetree_cover_obs_np[nonetree_cover_obs_np > 1e6] = np.nan

    # Common nan grid
    common_nan_grid_nonetree_cover = np.logical_or(np.all(np.isnan(nonetree_cover_jules_hist_np), axis = 0), np.all(np.isnan(nonetree_cover_obs_np), axis = 0))
    nonetree_cover_jules_hist_np[:, common_nan_grid_nonetree_cover] = np.nan
    nonetree_cover_obs_np[:, common_nan_grid_nonetree_cover] = np.nan

    print(f"--------Bias adjusting region {region}----------\n")

    for scenario in tqdm(scenarios):
        for model in models:
            try:
                if scenario == "historical":
                    period = "period_1994_2014"
                else:
                    period = "period_2015_2099"

                # Read in values
                filename = os.path.join("data", region, "isimp3b", scenario, model, period, "nonetree_cover_jules-es.nc")
                nonetree_cover_cm_fut = xr.open_dataset(filename)
                
                nonetree_cover_cm_fut_np = nonetree_cover_cm_fut.nonetree_cover.values
                nonetree_cover_cm_fut_np[nonetree_cover_cm_fut_np > 1e6] = np.nan

                # Get common nan mask (land-sea mask)
                nan_grid_cm_future = np.all(np.isnan(nonetree_cover_cm_fut_np), axis = 0)

                # Get values in cm future where values exist, but the common mask indicates nan (to infill later)
                cm_future_not_nan_but_obs_or_cm_hist = np.logical_and(np.logical_not(nan_grid_cm_future), common_nan_grid_nonetree_cover)

                # Map to common nan mask
                nonetree_cover_cm_fut_np_common_nan_grid = copy.deepcopy(nonetree_cover_cm_fut_np)
                nonetree_cover_cm_fut_np_common_nan_grid[:, common_nan_grid_nonetree_cover] = np.nan

                # Apply bias correction
                debiaser = LinearScaling(delta_type = "additive")
                debiased_nonetree_cover_cm_fut_common_nan_grid = debiaser.apply(nonetree_cover_obs_np, nonetree_cover_jules_hist_np, nonetree_cover_cm_fut_np_common_nan_grid,  progressbar = False, failsafe = True)
            
                # Reinfill values in cm_future outside of common nan grid 
                debiased_nonetree_cover_cm_fut = debiased_nonetree_cover_cm_fut_common_nan_grid
                debiased_nonetree_cover_cm_fut[:, cm_future_not_nan_but_obs_or_cm_hist] = nonetree_cover_cm_fut_np[:, cm_future_not_nan_but_obs_or_cm_hist] 

                # Map values below 0 and above 100 to min, max of obs:
                debiased_nonetree_cover_cm_fut[debiased_nonetree_cover_cm_fut < 0] = nonetree_cover_obs_np[~np.isnan(nonetree_cover_obs_np)].min()
                debiased_nonetree_cover_cm_fut[debiased_nonetree_cover_cm_fut > 100] = nonetree_cover_obs_np[~np.isnan(nonetree_cover_obs_np)].max()

                # Insert back into array
                nonetree_cover_cm_fut.nonetree_cover.values = debiased_nonetree_cover_cm_fut

                # Write to file
                nonetree_cover_cm_fut.to_netcdf(os.path.join("data", region, "isimp3b", scenario, model, period, "debiased_nonetree_cover_jules-es.nc"))


            except Exception as e:
                print(region, scenario, model)
                print(e)

--------Bias adjusting region Alberta----------



  0%|                                                     | 0/4 [00:00<?, ?it/s]/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: obs contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_hist contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_future contains inf or nan values. Not all debiasers support missing valu

--------Bias adjusting region Amazon----------



  0%|                                                     | 0/4 [00:00<?, ?it/s]/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: obs contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_hist contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_future contains inf or nan values. Not all debiasers support missing valu

--------Bias adjusting region Congo----------



...

  0%|                                                     | 0/4 [00:00<?, ?it/s]/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: obs contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_hist contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_future contains inf or nan values. Not all debiasers support missing valu

--------Bias adjusting region NWIndia----------



  0%|                                                     | 0/4 [00:00<?, ?it/s]/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: obs contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_hist contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_future contains inf or nan values. Not all debiasers support missing valu

--------Bias adjusting region Pantanal----------



  0%|                                                     | 0/4 [00:00<?, ?it/s]/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: obs contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_hist contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_future contains inf or nan values. Not all debiasers support missing valu

isimip3a, obsclim or counterclim, GSWP3-W5E5:

In [39]:
for region in regions:
    
    nonetree_cover_jules_hist = xr.open_dataset(f"data/{region}/isimp3a/obsclim/GSWP3-W5E5/period_2002_2019/nonetree_cover_jules-es.nc")
    nonetree_cover_obs = xr.open_dataset(f"data/{region}/isimp3a/obsclim/GSWP3-W5E5/period_2002_2019/nontree_raw_VCF-obs.nc")
    
    # Extract values
    nonetree_cover_jules_hist_np = nonetree_cover_jules_hist.nonetree_cover.values
    nonetree_cover_obs_np = nonetree_cover_obs.variable.values
    
    # Convert NaNs to np.nan (sometimes these are casted to high values)
    nonetree_cover_jules_hist_np[nonetree_cover_jules_hist_np > 1e6] = np.nan
    nonetree_cover_obs_np[nonetree_cover_obs_np > 1e6] = np.nan

    # Common nan grid
    common_nan_grid_nonetree_cover = np.logical_or(np.all(np.isnan(nonetree_cover_jules_hist_np), axis = 0), np.all(np.isnan(nonetree_cover_obs_np), axis = 0))
    nonetree_cover_jules_hist_np[:, common_nan_grid_nonetree_cover] = np.nan
    nonetree_cover_obs_np[:, common_nan_grid_nonetree_cover] = np.nan

    print(f"--------Bias adjusting region {region}----------\n")

    for clim_type in ["obsclim", "counterclim"]:
        for period in ["period_2000_2019", "period_1901_1920"]:
            
            try:
                
                # Read in values
                filename = os.path.join("data", region, "isimp3a", clim_type, "GSWP3-W5E5", period, "nonetree_cover_jules-es.nc")
                nonetree_cover_cm_fut = xr.open_dataset(filename)
                
                # Cut to the right area
                if region in ["MED", "Chile"]:
                    nonetree_cover_cm_fut = nonetree_cover_cm_fut.sel(
                        lat=slice(max(obs_latitudes), min(obs_latitudes)),
                        lon=slice(min(obs_longitudes), max(obs_longitudes)),
                    )
                
                nonetree_cover_cm_fut_np = nonetree_cover_cm_fut.nonetree_cover.values
                nonetree_cover_cm_fut_np[nonetree_cover_cm_fut_np > 1e6] = np.nan

                # Get common nan mask (land-sea mask)
                nan_grid_cm_future = np.all(np.isnan(nonetree_cover_cm_fut_np), axis = 0)

                # Get values in cm future where values exist, but the common mask indicates nan (to infill later)
                cm_future_not_nan_but_obs_or_cm_hist = np.logical_and(np.logical_not(nan_grid_cm_future), common_nan_grid_nonetree_cover)

                # Map to common nan mask
                nonetree_cover_cm_fut_np_common_nan_grid = copy.deepcopy(nonetree_cover_cm_fut_np)
                nonetree_cover_cm_fut_np_common_nan_grid[:, common_nan_grid_nonetree_cover] = np.nan

                # Apply bias correction
                debiaser = LinearScaling(delta_type = "additive")
                debiased_nonetree_cover_cm_fut_common_nan_grid = debiaser.apply(nonetree_cover_obs_np, nonetree_cover_jules_hist_np, nonetree_cover_cm_fut_np_common_nan_grid,  progressbar = False, failsafe = True)
            
                # Reinfill values in cm_future outside of common nan grid 
                debiased_nonetree_cover_cm_fut = debiased_nonetree_cover_cm_fut_common_nan_grid
                debiased_nonetree_cover_cm_fut[:, cm_future_not_nan_but_obs_or_cm_hist] = nonetree_cover_cm_fut_np[:, cm_future_not_nan_but_obs_or_cm_hist] 

                # Map values below 0 and above 100 to min, max of obs:
                debiased_nonetree_cover_cm_fut[debiased_nonetree_cover_cm_fut < 0] = nonetree_cover_obs_np[~np.isnan(nonetree_cover_obs_np)].min()
                debiased_nonetree_cover_cm_fut[debiased_nonetree_cover_cm_fut > 100] = nonetree_cover_obs_np[~np.isnan(nonetree_cover_obs_np)].max()

                # Insert back into array
                nonetree_cover_cm_fut.nonetree_cover.values = debiased_nonetree_cover_cm_fut

                # Write to file
                nonetree_cover_cm_fut.to_netcdf(os.path.join("data", region, "isimp3a", clim_type, "GSWP3-W5E5", period, "debiased_nonetree_cover_jules-es.nc"))

            except Exception as e:
                print(region, clim_type, period)
                print(e)

--------Bias adjusting region Alberta----------

--------Bias adjusting region Amazon----------



/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: obs contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_hist contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_future contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. 

--------Bias adjusting region Congo----------



/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: obs contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_hist contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_future contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. 

--------Bias adjusting region LA----------

--------Bias adjusting region NWIndia----------



/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: obs contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_hist contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_future contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. 

--------Bias adjusting region Pantanal----------



/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: obs contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_hist contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. Consider infilling the missing values.
  obs, cm_hist, cm_future = self._check_inputs_and_convert_if_possible(
/Users/fionaspuler/opt/anaconda3/lib/python3.9/site-packages/ibicus/debias/_debiaser.py:532: UserWarning: cm_future contains inf or nan values. Not all debiasers support missing values and their presence might lead to infs or nans inside of the debiased values. 